# 02 — Entity Resolution & Linking

**Project:** Predicting Corporate GHG Intensity  
**Purpose:** Match EPA facility parent names to SEC corporate entities and construct the panel.

---


In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.entity_resolution import EntityResolver

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

### 1. Load Raw Datasets

In [2]:
df_epa = pd.read_csv(RAW_DIR / "epa_ghgrp_facilities.csv")
df_sec = pd.read_csv(RAW_DIR / "sec_financials.csv")
df_wb = pd.read_csv(RAW_DIR / "worldbank_macro.csv")

sec_index = df_sec[["cik", "company_name"]].drop_duplicates(subset=["cik"])
print(f"Loaded {len(df_epa)} EPA facility rows and {len(df_sec)} SEC firm-year rows.")

Loaded 39230 EPA facility rows and 2791 SEC firm-year rows.


### 2. Run Fuzzy Entity Resolution
We match EPA parent companies against the SEC corporate index.

In [3]:
resolver = EntityResolver(output_dir=INTERIM_DIR)
mapping_df = resolver.resolve_entities(df_epa, sec_index, fuzzy_threshold=70)
mapping_df.head(10)

2026-06-29 16:10:40,576 - INFO - Resolving 4443 unique EPA parents against 657 SEC companies …


2026-06-29 16:10:45,142 - INFO - Entity resolution complete: 2189 resolved (229 exact + 1960 fuzzy), 2254 unresolved → E:\Research_Projects\predicting-corporate-ghg-intensity\data\interim\epa_sec_mapping.csv


,epa_parent,sec_company_name,cik,ticker,similarity_score,is_resolved
0,NORTH TEXAS MUNICIPAL WATER DISTRICT,,,,0.000000,False
1,CAMBRIAN COAL LLC,,,,0.000000,False
2,VERTELLUS HOLDINGS LLC,,,,0.000000,False
3,NEW YORK POWER AUTHORITY,THE NEW YORK TIMES COMPANY,71691,,72.727273,True
4,ARCHDIOCESE OF CHICAGO,Chicago Rivet & Machine Co.,19871,,70.000000,True
5,3M CO,3M COMPANY,66740,,100.000000,True
6,MEPCO INTERMEDIATE HOLDINGS LLC,,,,0.000000,False
7,HOLLAND BOARD OF PUBLIC WORKS,,,,0.000000,False
8,MIAMI-DADE COUNTY DEPARTMENT OF SOLID WASTE MA...,GEX MANAGEMENT INC.,1681556,,83.333333,True
9,CONSOLIDATED EDISON INC,CONSOLIDATED EDISON INC,1047862,,100.000000,True


### 3. Link Datasets into a Panel
This merges reporting and non-reporting companies, filling missing emissions with NaN to preserve the full selection process.

In [4]:
control_ciks = [c for c in df_sec["cik"].unique() if c not in mapping_df[mapping_df["is_resolved"]]["cik"].values]
linked_df = resolver.link_datasets(
    df_epa, df_sec, mapping_df, df_wb,
    control_ciks=control_ciks
)
print("Linked panel shape:", linked_df.shape)
linked_df.head()

2026-06-29 16:10:45,477 - INFO - Linked panel: 2791 firm-years (657 unique firms), 1784 reporting, 1007 non-reporting → E:\Research_Projects\predicting-corporate-ghg-intensity\data\interim\linked_panel.csv


Linked panel shape: (2791, 23)


,cik,ticker,company_name,sector,sic_code,year,total_assets,revenue,net_income,operating_income,...,stockholders_equity,scope1_emissions,co2_non_biogenic,n_facilities,primary_naics,high_emission_naics,selected,us_gdp_growth,us_co2_per_capita,us_energy_use_per_capita
0,19871,CVR,Chicago Rivet & Machine Co.,Manufacturing,3540,2018,33246625.0,37174249.0,2001185.0,2402648.0,...,29759749.0,55180.90,168.4,2.0,562212.0,0,1,2.966505,15.2,6738.272724
1,19871,CVR,Chicago Rivet & Machine Co.,Manufacturing,3540,2019,31723376.0,32873002.0,538314.0,491584.0,...,29158027.0,30150.65,246.9,2.0,562212.0,0,1,2.583825,14.8,6700.930340
2,19871,CVR,Chicago Rivet & Machine Co.,Manufacturing,3540,2020,31238071.0,27590653.0,50450.0,-83014.0,...,28706089.0,NaN,NaN,NaN,NaN,0,0,-2.163029,13.0,6141.531258
3,19871,CVR,Chicago Rivet & Machine Co.,Manufacturing,3540,2021,31766258.0,33974558.0,1113472.0,1358915.0,...,28969365.0,NaN,NaN,NaN,NaN,0,0,6.055053,13.9,6445.834092
4,19871,CVR,Chicago Rivet & Machine Co.,Manufacturing,3540,2022,33626127.0,33646033.0,2867629.0,3561196.0,...,30986798.0,NaN,NaN,NaN,NaN,0,0,2.512375,13.6,6511.688414


### Discussion & Next Steps
The entity resolver successfully linked the EPA parents to SEC CIKs. The resulting `linked_panel.csv` preserves both selection groups (selected = 1 and selected = 0), which is essential for econometric selection bias correction. Next, we will perform a data quality audit.